In [ ]:
import pandas as pd

In [2]:
#previous dfs
draft_df = pd.read_csv('/Users/indigokhanna/Documents/GitHub/Draft-Decisions-Casual-Inference/Inputs/54WinRateData.csv')
ALSA = pd.read_csv('../Outputs/FDN_ALSA.csv')
ALSA.head()
card_df = pd.read_csv('../Inputs/FDNCardData.csv')
card_df = card_df.drop('ALSA', axis=1)

df_merged = pd.merge(card_df, ALSA[['pick', 'ALSA']], left_on='Name',right_on = 'pick' ,how='left')
df_merged = df_merged.drop('pick', axis=1)


In [3]:
#The functions for getting pack characteristics
basics = ['Plains','Island','Swamp','Mountain','Forest']
pack_col_prefix = 'pack_card_'

def get_cards_in_pack(row):
    '''Get cards from row, return their name, color, and ALSA'''
    card_mask = row.filter(like='pack_card_') == 1
    cards = [col[10:] for col in card_mask[card_mask].index]
    #cards = [col[10:] for col in row.index[row == 1]]
    cards_in_pack_df = df_merged[df_merged['Name'].isin(cards) & ~df_merged['Name'].isin(basics)][['Name', 'Color', 'ALSA']]
    return cards_in_pack_df

def get_pack_color_metrics(row):
    '''Get cards from row, get card data, group by color, take the min and mean ALSA'''
    cards_in_pack_df = get_cards_in_pack(row)
    mins = {f'{color}_min': val for color, val in cards_in_pack_df.groupby('Color')['ALSA'].min().items()}
    means = {f'{color}_mean': val for color, val in cards_in_pack_df.groupby('Color')['ALSA'].mean().items()}
    return {**mins, **means}

#and applying them to the alsa
color_columns = draft_df.apply(get_pack_color_metrics, axis=1, result_type='expand')
draft_df_with_ALSA = pd.concat([draft_df, color_columns], axis=1)


In [4]:
#functions for determining color identity 

def get_previous_picks(row):
    '''get and return previous rows from the same draft'''
    current_pack = row['pack_number']
    current_pick = row['pick_number']
    draft = draft_df[draft_df['draft_id'] == row['draft_id']]
    
    previous_packs = draft[draft['pack_number'] < current_pack]
    current_pack_previous_picks = draft[
        (draft['pack_number'] == current_pack) & 
        (draft['pick_number'] < current_pick)
    ]
    
    return pd.concat([previous_packs, current_pack_previous_picks])

def get_color_identity(row):
    '''gets rows before given row and returns the top two colors from previously picked cards'''
    previous_picks = get_previous_picks(row)['pick']
    cards_in_pack_df = df_merged[df_merged['Name'].isin(previous_picks) & ~df_merged['Name'].isin(basics)][['Name', 'Color', 'ALSA']]
    
    colors = cards_in_pack_df['Color'].dropna().apply(list).explode()
    top_two_colors = colors.value_counts().index[:2].tolist()

    return top_two_colors

#add color identity column to data frame
draft_df_with_ALSA['color_identity'] = draft_df_with_ALSA.apply(get_color_identity, axis=1)

In [5]:
#function to assign treatment

color_mean_ALSA_cols = set(draft_df_with_ALSA.filter(like='mean'))

def assign_treatment(row, treatment_threshold, pick_limit=100):
    '''Gets ALSA colors, subtracts the current pick number, returns true if that value is below treatment threshold and outside colors'''

    cards = get_cards_in_pack(row)
    pool_colors = row['color_identity']
    
    #return false if empty
    if cards.empty:
        return {'treatment': False, 'instrument': False}
    
    #limit treatment just to picks before pick_limit
    if row['pick_number'] >= pick_limit:
        return {'treatment': False, 'instrument': False}
    
    #we only want treatment drafts where we already have two colors of picks
    if len(pool_colors) < 2:
        return {'treatment': False, 'instrument': False}

    #Do pick asdjusted ALSA over pack, remove colorless cards, then store cards below treatment threshold as best_cards
    cards['PAA'] = cards['ALSA'].apply(lambda x: x - (row['pick_number'] + 1))
    best_cards = cards[(abs(cards['PAA']) <= treatment_threshold) & (cards['Color'].notna())]

    #of best cards, check which aren't in our colors. These are our treatment cards
    best_cards['in_colors'] = best_cards['Color'].apply(lambda x: all(c in pool_colors for c in x))
    treatment_cards = best_cards[best_cards['in_colors'] == False]

    picked_treatment_card = treatment_cards['Name'].isin([row['pick']]).any()

    #treatment is if they picked a treatment card
    treatment = picked_treatment_card
    #instrument is if they saw a treatment card but didn't take it
    instrument = not treatment_cards.empty
    
    return {'treatment': treatment, 'instrument': instrument, 'best_cards': treatment_cards['Name']}

In [6]:

#Add treatment to the data frame, pick limit = 10

#this is whether a 

treatment_threshold = 4 #4 is probably fine as a threshold

draft_df_with_ALSA[['treatment','instrument','best_cards']] = draft_df_with_ALSA.apply(lambda row: assign_treatment(row,treatment_threshold,pick_limit=10), axis=1, result_type='expand')


In [7]:
#for formatting purposes
draft_df_with_ALSA[draft_df_with_ALSA['treatment'] == True]


,Unnamed: 0,expansion,event_type,draft_id,draft_time,rank,event_match_wins,event_match_losses,pack_number,pick_number,...,G_min,G_mean,WUBRG_min,WUBRG_mean,W_min,W_mean,color_identity,treatment,instrument,best_cards
87,88,FDN,TradDraft,6e429fc7ba9a4b9fb58cc4186b66ab09,2024-11-12 22:26:31,NaN,2,1,0,3,...,4.821894,4.821894,NaN,NaN,NaN,NaN,"[U, W]",True,True,172 Exsanguinate 229 Overrun Name: ...
202,203,FDN,TradDraft,7e8c5392fcd5405888893729294964ac,2024-11-12 18:37:44,NaN,1,2,2,6,...,5.017861,5.017861,NaN,NaN,NaN,NaN,"[B, R]",True,True,"214 Bushwhack Name: Name, dtype: object"
315,316,FDN,TradDraft,eb8d939c6607408cbf2a4725634e93f7,2024-11-12 17:55:23,NaN,2,1,1,7,...,NaN,NaN,NaN,NaN,NaN,NaN,"[W, G]",True,True,"47 Refute 180 Pilfer Name: Name, dtype:..."
704,705,FDN,TradDraft,c5479925aa51415d85901fa184ba5df1,2024-11-12 21:54:11,NaN,3,0,2,4,...,NaN,NaN,5.856931,5.856931,NaN,NaN,"[U, B]",True,True,"243 Progenitus Name: Name, dtype: object"
1096,1097,FDN,TradDraft,3549f31ee70b410b9aa9d24b2e6ac515,2024-11-12 18:38:21,NaN,2,1,0,4,...,NaN,NaN,NaN,NaN,NaN,NaN,"[R, W]",True,True,"150 Aetherize Name: Name, dtype: object"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55554,55555,FDN,TradDraft,2a71e0f82c194e8582cffea985d92737,2024-12-14 18:01:54,NaN,1,2,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,"[W, U]",True,True,"70 Stab Name: Name, dtype: object"
55737,55738,FDN,TradDraft,6f9b2d1c4fb3491e81c04d8c657b238d,2024-12-14 19:52:44,NaN,1,2,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,"[W, U]",True,True,"187 Abrade Name: Name, dtype: object"
55779,55780,FDN,TradDraft,1c6b839063a547758ecc02b7e0e13a14,2024-12-14 20:00:46,NaN,2,1,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,"[B, R]",True,True,"47 Refute Name: Name, dtype: object"
56161,56162,FDN,TradDraft,82bf9e44c63b42e0b5c88a1df576b68d,2024-12-15 16:52:21,NaN,1,2,1,7,...,NaN,NaN,NaN,NaN,NaN,NaN,"[B, U]",True,True,"84 Electroduplicate Name: Name, dtype: object"


In [ ]:
#set treatment pack and pick
#looping through all the packs and picks (after p1p5) and storing the treatment
#store the treatment, and a col for each pack and pick in the draft and whether the treatment happened there
#i'll be real idk why we're doing this by pack and pick, but sure
treat_pack = 0
treat_pick = 5


#get treatment and instrument rows, save the draft ids to join on after matching
treatment_df = draft_df_with_ALSA.query('pack_number == @treat_pack and pick_number == @treat_pick and treatment')
treatment_and_instrument_df = draft_df_with_ALSA.query('pack_number == @treat_pack and pick_number == @treat_pick and treatment and instrument')

treatment_draft_ids = list(set(treatment_df['draft_id']))
treatment_and_instrument_draft_ids = list(set(treatment_and_instrument_df['draft_id']))

control_df = draft_df_with_ALSA[~draft_df_with_ALSA['draft_id'].isin(treatment_draft_ids)]

#now do that for all the packs and picks after p1p5 and concat the results
#the first one is slightly different bc we're starting at p5
for treat_pick in range(6,16):
    cur_treatment_df = draft_df_with_ALSA.query('pack_number == @treat_pack and pick_number == @treat_pick and treatment')
    treatment_df=pd.concat([treatment_df,cur_treatment_df])
    cur_treatment_and_instrument_df = draft_df_with_ALSA.query('pack_number == @treat_pack and pick_number == @treat_pick and treatment and instrument')
    treatment_and_instrument_df=pd.concat([treatment_and_instrument_df,cur_treatment_and_instrument_df])
    #for these also concat them with the rest of the packs/picks
    cur_treatment_draft_ids = list(set(cur_treatment_df['draft_id']))
    treatment_draft_ids=treatment_draft_ids+cur_treatment_draft_ids

    cur_treatment_and_instrument_draft_ids = list(set(cur_treatment_and_instrument_df['draft_id']))
    treatment_and_instrument_draft_ids=treatment_and_instrument_draft_ids+cur_treatment_and_instrument_draft_ids

    cur_control_df = draft_df_with_ALSA[~draft_df_with_ALSA['draft_id'].isin(cur_treatment_draft_ids)]
    control_df=pd.concat([control_df,cur_control_df])
    
#packs 2 and 3
for treat_pack in range(1,3):
    for treat_pick in range(0,16):

        cur_treatment_df = draft_df_with_ALSA.query('pack_number == @treat_pack and pick_number == @treat_pick and treatment')
        treatment_df=pd.concat([treatment_df,cur_treatment_df])
        cur_treatment_and_instrument_df = draft_df_with_ALSA.query('pack_number == @treat_pack and pick_number == @treat_pick and treatment and instrument')
        treatment_and_instrument_df=pd.concat([treatment_and_instrument_df,cur_treatment_and_instrument_df])
        #for these also concat them with the rest of the packs/picks
        cur_treatment_draft_ids = list(set(cur_treatment_df['draft_id']))
        treatment_draft_ids=treatment_draft_ids+cur_treatment_draft_ids #this is a list

        cur_treatment_and_instrument_draft_ids = list(set(cur_treatment_and_instrument_df['draft_id']))
        treatment_and_instrument_draft_ids=treatment_and_instrument_draft_ids+cur_treatment_and_instrument_draft_ids

        cur_control_df = draft_df_with_ALSA[~draft_df_with_ALSA['draft_id'].isin(cur_treatment_draft_ids)]
        control_df=pd.concat([control_df,cur_control_df]) #this is a df

In [ ]:
#write the outputs to files (convert the lists to dfs)
treatment_draft_ids = pd.DataFrame(treatment_draft_ids, columns=['Draft_id'])
treatment_draft_ids.to_csv('../Outputs/treatment_draft_ids.csv')

treatment_and_instrument_draft_ids = pd.DataFrame(treatment_and_instrument_draft_ids, columns=['Draft_id'])
treatment_and_instrument_draft_ids.to_csv('../Outputs/treatment_and_instrument_draft_ids.csv')

control_df.to_csv('../Outputs/control_df.csv')
treatment_df.to_csv('../Outputs/treatment_df.csv')
treatment_and_instrument_df.to_csv('../Outputs/treatment_and_instrument_df.csv')

In [ ]:
treatment_and_instrument_df.tail(10)

,Unnamed: 0,expansion,event_type,draft_id,draft_time,rank,event_match_wins,event_match_losses,pack_number,pick_number,...,G_min,G_mean,WUBRG_min,WUBRG_mean,W_min,W_mean,color_identity,treatment,instrument,best_cards
46307,46308,FDN,TradDraft,d8fd8dc2618245c1873feb66e89ea19e,2024-12-03 15:04:58,NaN,0,3,2,8,...,NaN,NaN,NaN,NaN,NaN,NaN,"[U, W]",True,True,"180 Pilfer Name: Name, dtype: object"
47004,47005,FDN,TradDraft,3785203195114bcc8b0303525b5f9932,2024-12-04 14:53:32,NaN,0,0,2,8,...,NaN,NaN,NaN,NaN,NaN,NaN,"[U, G]",True,True,"180 Pilfer Name: Name, dtype: object"
51286,51287,FDN,TradDraft,a1f9c5712ffa4f03a4724462a42035fe,2024-12-07 17:24:00,NaN,1,2,2,8,...,NaN,NaN,5.856931,5.856931,NaN,NaN,"[W, B]",True,True,"243 Progenitus Name: Name, dtype: object"
4317,4318,FDN,TradDraft,c4b51e6a261a427facfd370a26cb9774,2024-11-14 02:17:09,NaN,2,1,2,9,...,NaN,NaN,NaN,NaN,NaN,NaN,"[W, B]",True,True,"84 Electroduplicate Name: Name, dtype: object"
4485,4486,FDN,TradDraft,27c2955e7ca44b6d8d92b64e8ceca2e5,2024-11-14 15:25:39,NaN,2,1,2,9,...,NaN,NaN,NaN,NaN,NaN,NaN,"[W, G]",True,True,"180 Pilfer Name: Name, dtype: object"
11868,11869,FDN,TradDraft,b7c128793a404d84b80c62b518f07cbd,2024-11-16 22:03:36,NaN,2,1,2,9,...,NaN,NaN,NaN,NaN,NaN,NaN,"[W, B]",True,True,"78 Boltwave Name: Name, dtype: object"
22283,22284,FDN,TradDraft,0254299767794fafa95e4653003665ce,2024-11-21 19:25:47,NaN,1,0,2,9,...,NaN,NaN,NaN,NaN,NaN,NaN,"[W, B]",True,True,"78 Boltwave Name: Name, dtype: object"
33114,33115,FDN,TradDraft,54105bb24f0f417daaf4e8ce9262af41,2024-11-26 22:33:35,NaN,1,1,2,9,...,NaN,NaN,NaN,NaN,NaN,NaN,"[R, G]",True,True,"172 Exsanguinate Name: Name, dtype: object"
53741,53742,FDN,TradDraft,860f790e7c644fb6b50c5e4113688a8d,2024-12-09 19:12:42,NaN,2,1,2,9,...,NaN,NaN,NaN,NaN,NaN,NaN,"[R, W]",True,True,"172 Exsanguinate Name: Name, dtype: object"
56470,56471,FDN,TradDraft,8e41afb2fa7248bfaa4009456774e87d,2024-12-15 19:06:36,NaN,2,1,2,9,...,5.017861,5.017861,NaN,NaN,NaN,NaN,"[U, W]",True,True,"78 Boltwave 180 Pilfer Name: Name, dt..."
